# installing dependencies

In [ ]:
!pip install -q emoji
!pip install -q PyArabic
!pip install -q arabert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.2 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets torch

In [ ]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

There are 1 GPU(s) available.
We will use the GPU: Tesla T4
Sat Apr  4 14:10:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |            

# Creating training datasets

In [ ]:
import pandas as pd
import numpy as np
from typing import List
from tqdm import tqdm_notebook as tqdm
from sklearn.model_selection import train_test_split

This custom dataset class will help us hold our datasets in a structred manner.
It's not necessary to use it with your own data

In [ ]:
class CustomDataset:
    def __init__(
        self,
        name: str,
        train: List[pd.DataFrame],
        val: List[pd.DataFrame],
        test: List[pd.DataFrame],
        label_list: List[str],
    ):
        """Class to hold and structure datasets.

        Args:

        name (str): holds the name of the dataset so we can select it later
        train (List[pd.DataFrame]): holds training pandas dataframe with 2 columns ["text","label"]
        val (List[pd.DataFrame]): holds validation pandas dataframe with 2 columns ["text","label"]
        test (List[pd.DataFrame]): holds testing pandas dataframe with 2 columns ["text","label"]
        label_list (List[str]): holds the list  of labels
        """
        self.name = name
        self.train = train
        self.val = val
        self.test = test
        self.label_list = label_list

In [ ]:
# This will hold all the downloaded and structred datasets
all_datasets= []
DATA_COLUMN = "Text"
LABEL_COLUMN = "sentiment"

You can choose which ever dataset you like or use your own.
At this stage we don't do any preprocessing on the text, this is done later when loading the text.

## Prepare Non labeled Climate Data

In [ ]:
all = pd.read_excel('All Climate Change Data - All Related.xlsx')

all.drop_duplicates(subset='text', inplace = True)
all.dropna(inplace = True, subset='text')
all.reset_index(drop=True, inplace = True)

all = all.rename(columns={'text':'Text'})

all = all[['Text', 'sentiment']]
all['sentiment'] = all['sentiment'].str.lower()

In [ ]:
all['Text'] = all['Text'].str.replace(r'[^\w\s]+', '')
all['Text'] = all['Text'].str.replace("\s+", " ", regex=True)

In [ ]:
import string

arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
import re

all['Text'] = all['Text'].apply(normalize_arabic)

In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
import pyarabic.araby as araby
import emoji

all[DATA_COLUMN] = data_cleaning(all[DATA_COLUMN])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

all['Text'] = all['Text'].apply(remove_ids)
all.head()

,Text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,neutral
2,@: عقد في مالطا هذا الاسبوع اول اجتماع لمجلس ا...,neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,neutral


In [ ]:
climate, unlabeled = train_test_split(all,
                               test_size=1/3,
                               stratify=all[LABEL_COLUMN],
                               random_state=42)

In [ ]:
unlabeled.dropna(inplace = True)
unlabeled = unlabeled.drop_duplicates(subset = DATA_COLUMN)
unlabeled.shape

(7986, 2)

In [ ]:
unlabeled[LABEL_COLUMN].value_counts()

,count
sentiment,
neutral,3436
negative,2713
positive,1837


## SE Data

In [ ]:
data = pd.read_csv('train_all.csv')
data = data[[DATA_COLUMN, LABEL_COLUMN]]

data.columns = [DATA_COLUMN, LABEL_COLUMN]
print(data[LABEL_COLUMN].value_counts())

sentiment
neutral     37359
positive     8821
negative     8820
Name: count, dtype: int64


In [ ]:
data.shape

(55000, 2)

In [ ]:
print(data[LABEL_COLUMN].value_counts())

sentiment
neutral     37359
positive     8821
negative     8820
Name: count, dtype: int64


In [ ]:
data[DATA_COLUMN] = data[DATA_COLUMN].str.replace(r'[^\w\s]+', '')
data[DATA_COLUMN] = data[DATA_COLUMN].str.replace("\s+", " ", regex=True)

data[DATA_COLUMN] = data[DATA_COLUMN].apply(normalize_arabic)
data[DATA_COLUMN] = data[DATA_COLUMN].apply(remove_ids)

data.head()

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_892/2797559622.py:2: SyntaxWarning: invalid escape sequence '\s'
  data[DATA_COLUMN] = data[DATA_COLUMN].str.replace("\s+", " ", regex=True)


,Text,sentiment
0,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...,neutral
1,@halgawi @DmfMohe ليس حباً في ايران بقدر ماهو ...,neutral
2,@adalfahadduwail ابي اعرف الحاكم العربي المسلم...,neutral
3,@sarmadbouchamou @DimaSadek في الخطاب تبع سليم...,neutral
4,@FofaMahmoudd مفيش الكلام ده في الزمن,neutral


In [ ]:
ind = pd.read_excel('/content/Used ASAD For Training.xlsx')

keep = data.loc[ind['Unnamed: 0'].values]
keep = keep[["Text", "sentiment"]]

keep.columns = ["Text", "sentiment"]
keep = keep.reset_index(drop = True)
print(keep["sentiment"].value_counts())

sentiment
neutral     16289
negative     3890
positive     3778
Name: count, dtype: int64


In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate-ASAD.csv')

# Clean and convert to integer index arrays
test = keep.loc[indecies['test'].dropna().astype(int).values]
train = keep.loc[indecies['train'].dropna().astype(int).values]
val = keep.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train['sentiment'].value_counts()

,count
sentiment,
neutral,11402
negative,2723
positive,2644


In [ ]:
label_list = ['negative', 'neutral', 'positive']

data = CustomDataset("Climate", train, val, test, label_list)
all_datasets.append(data)

#Trainer

Start the training procedure

In [ ]:
import numpy as np
import torch
import random
import matplotlib.pyplot as plt
import copy

from arabert.preprocess import ArabertPreprocessor
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoConfig, AutoModelForSequenceClassification,
                          AutoTokenizer, BertTokenizer, Trainer,
                          TrainingArguments)
from transformers.data.processors.utils import InputFeatures

List all the datasets we have

In [ ]:
for x in all_datasets:
  print(x.name)

Climate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# select a dataset
dataset_name = 'Climate'
model_name = 'aubmindlab/bert-base-arabertv02-twitter'

In [ ]:
for d in all_datasets:
  if d.name==dataset_name:
    selected_dataset = copy.deepcopy(d)
    print('Dataset found')
    break

Dataset found


Create and apply preprocessing using the AraBERT processor

In [ ]:
arabic_prep = ArabertPreprocessor(model_name)

unlabeled['Text'] = unlabeled['Text'].apply(lambda x: arabic_prep.preprocess(x))

selected_dataset.train[DATA_COLUMN] = selected_dataset.train[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))
selected_dataset.val[DATA_COLUMN] = selected_dataset.val[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))
selected_dataset.test[DATA_COLUMN] = selected_dataset.test[DATA_COLUMN].apply(lambda x: arabic_prep.preprocess(x))

Now we need to check the tokenized sentence length to decide on the maximum sentence length value

In [ ]:
tok = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/476 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Let's select 100 as our maximum sentence length, and check how many sequences will be truncated

In [ ]:
max_len = 128

8 out of ~4000 for testing isn't bad

Now let's create a classification dataset to load the data

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, text, target, model_name, max_len, label_map):
      super(ClassificationDataset).__init__()
      """
      Args:
      text (List[str]): List of the training text
      target (List[str]): List of the training labels
      tokenizer_name (str): The tokenizer name (same as model_name).
      max_len (int): Maximum sentence length
      label_map (Dict[str,int]): A dictionary that maps the class labels to integer
      """
      self.text = text
      self.target = target
      self.tokenizer_name = model_name
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.max_len = max_len
      self.label_map = label_map


    def __len__(self):
      return len(self.text)

    def __getitem__(self,item):
      text = str(self.text[item])
      text = " ".join(text.split())

      inputs = self.tokenizer(
          text,
          max_length=self.max_len,
          padding='max_length',
          truncation=True
      )
      return InputFeatures(**inputs,label=self.label_map[self.target[item]])

In [ ]:
label_map = { v:index for index, v in enumerate(selected_dataset.label_list) }
print(label_map)

train_dataset = ClassificationDataset(
    selected_dataset.train[DATA_COLUMN].to_list(),
    selected_dataset.train[LABEL_COLUMN].to_list(),
    model_name,
    max_len,
    label_map
  )
val_dataset = ClassificationDataset(
    selected_dataset.val[DATA_COLUMN].to_list(),
    selected_dataset.val[LABEL_COLUMN].to_list(),
    model_name,
    max_len,
    label_map
  )

{'negative': 0, 'neutral': 1, 'positive': 2}


In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

# Convert your pandas DataFrame to Hugging Face dataset
hf_dataset = Dataset.from_pandas(unlabeled[['Text']])  # Only keep needed column

# Tokenize efficiently using map
def tokenize_function(examples):
    return tok(examples['Text'], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

# Prepare the MLM collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)

Map:   0%|          | 0/7986 [00:00<?, ? examples/s]

Create a function that return a pretrained model ready to do classification

In [ ]:
from transformers import AutoModelForMaskedLM

def model_init():
    return AutoModelForMaskedLM.from_pretrained(model_name)

In [ ]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic=True
  torch.backends.cudnn.benchmark = False

#Regular Training

Define our training parameters.
Check the TrainingArguments documentation for more options https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./arabert-mlm",
    eval_strategy="no",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    save_steps=10_000,
    save_total_limit=2,
    report_to='none'
)

set_seed(training_args.seed)

Create the trainer

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model_init(),
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

In [ ]:
#start the training
trainer.train()

Save the model, the tokenizer and the config

In [ ]:
dir = '/content/drive/MyDrive/DAPT-ASAD'
trainer.save_model(dir)
tok.save_pretrained(dir)

# Train on ASAD

In [ ]:
model_name2 = dir
def model_init2():
    return AutoModelForSequenceClassification.from_pretrained(model_name2, return_dict=True, num_labels=len(label_map))

In [ ]:
def compute_metrics(p): #p should be of type EvalPrediction
  preds = np.argmax(p.predictions, axis=1)
  assert len(preds) == len(p.label_ids)
  macro_f1 = f1_score(p.label_ids,preds,average='macro')
  macro_precision = precision_score(p.label_ids,preds,average='macro')
  macro_recall = recall_score(p.label_ids,preds,average='macro')
  acc = accuracy_score(p.label_ids,preds)
  return {
      'macro_f1' : macro_f1,
      'macro_precision' : macro_precision,
      'macro_recall' : macro_recall,
      'accuracy': acc
  }

In [ ]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic=True
  torch.backends.cudnn.benchmark = False

In [ ]:
training_args = TrainingArguments(
    output_dir= "./train",
    adam_epsilon = 1e-8,
    learning_rate = 2e-5,
    fp16 = False, # enable this when using V100 or T4 GPU
    per_device_train_batch_size = 32, # up to 64 on 16GB with max len of 128
    per_device_eval_batch_size = 128,
    gradient_accumulation_steps = 2, # use this to scale batch size without needing more memory
    num_train_epochs= 5,
    warmup_ratio = 0,
    do_eval = True,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True, # this allows to automatically get the best model at the end based on whatever metric we want
    metric_for_best_model = 'macro_f1',
    greater_is_better = True,
    seed = 25,
    report_to='none'
  )

set_seed(training_args.seed)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer2 = Trainer(
    model = model_init2(),
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
#start the training
trainer2.train()

Epoch,Training Loss,Validation Loss,Macro F1,Macro Precision,Macro Recall,Accuracy
1,No log,0.361069,0.803103,0.798802,0.812415,0.848637
2,0.723190,0.373853,0.805283,0.788815,0.825444,0.845019
3,0.723190,0.411034,0.806642,0.791797,0.823954,0.847802
4,0.402676,0.441794,0.802285,0.786557,0.820636,0.843072
5,0.402676,0.463310,0.803586,0.792578,0.815808,0.845854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1315, training_loss=0.4958143923219166, metrics={'train_runtime': 1905.9564, 'train_samples_per_second': 43.991, 'train_steps_per_second': 0.69, 'total_flos': 5515186127351040.0, 'train_loss': 0.4958143923219166, 'epoch': 5.0})

In [ ]:
inv_label_map = inv_label_map = { v:k for k, v in label_map.items()}
print(inv_label_map)
trainer2.model.config.label2id = label_map
trainer2.model.config.id2label = inv_label_map
trainer2.save_model("pre-trainedASAD")
train_dataset.tokenizer.save_pretrained("pre-trainedASAD")

{0: 'negative', 1: 'neutral', 2: 'positive'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('pre-trainedASAD/tokenizer_config.json', 'pre-trainedASAD/tokenizer.json')

## predict using the saved model

In [ ]:
from transformers import pipeline
import more_itertools

In [ ]:
pred_df = pd.DataFrame()
pred_df['text'] = selected_dataset.test[DATA_COLUMN].values

In [ ]:
pipe = pipeline("sentiment-analysis", model=f"pre-trainedASAD", device=0, return_all_scores =True, max_length=max_len, truncation=True)
preds = []
for s in tqdm(more_itertools.chunked(list(pred_df['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_df[f'preds'] = preds

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_892/4222897334.py:3: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for s in tqdm(more_itertools.chunked(list(pred_df['text']), 32)): # batching for faster inference


0it [00:00, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
final_pred = []
for prediction in pred_df['preds']:
  final_pred.append(prediction['label'])

pred_df[f'Final Prediction'] = final_pred
pred_df[f'Final Prediction'].value_counts()

,count
Final Prediction,
neutral,2342
positive,639
negative,613


In [ ]:
all_datasets[0].test[LABEL_COLUMN].value_counts()

,count
sentiment,
neutral,2443
negative,584
positive,567


In [ ]:
print(classification_report(all_datasets[0].test[LABEL_COLUMN],pred_df['Final Prediction'], digits=4))

              precision    recall  f1-score   support

    negative     0.7129    0.7483    0.7302       584
     neutral     0.9133    0.8756    0.8940      2443
    positive     0.7746    0.8730    0.8209       567

    accuracy                         0.8545      3594
   macro avg     0.8003    0.8323    0.8150      3594
weighted avg     0.8589    0.8545    0.8559      3594



# Pred on Collected

In [ ]:
climate.dropna(inplace = True)
climate = climate.drop_duplicates(subset = 'Text')
climate.shape

(15969, 2)

In [ ]:
pred_climate = pd.DataFrame()
pred_climate['text'] = climate['Text'].values

pipe = pipeline("sentiment-analysis", model=f"pre-trainedASAD", device=0, return_all_scores =True, max_length=max_len, truncation=True)

preds = []
for s in tqdm(more_itertools.chunked(list(pred_climate['text']), 32)): # batching for faster inference
    preds.extend(pipe(s))
pred_climate[f'preds'] = preds

final_pred = []
for prediction in pred_climate['preds']:
  final_pred.append(prediction['label'])

pred_climate[f'Final Prediction'] = final_pred
pred_climate[f'Final Prediction'].value_counts()

,count
Final Prediction,
neutral,13785
negative,1445
positive,739


In [ ]:
climate['sentiment'].value_counts()

,count
sentiment,
neutral,6871
negative,5427
positive,3673


In [ ]:
climate['Final Label'] = climate['sentiment'].str.lower()
print(classification_report(climate['Final Label'],pred_climate['Final Prediction'], digits=4))

              precision    recall  f1-score   support

    negative     0.8098    0.2165    0.3417      5427
     neutral     0.4662    0.9348    0.6221      6871
    positive     0.6797    0.1375    0.2287      3673

    accuracy                         0.5074     15971
   macro avg     0.6519    0.4296    0.3975     15971
weighted avg     0.6321    0.5074    0.4364     15971

